# tmp

# Datamart Month Acquiring — FinalDF + Excel Compare (декомпозиция)

Тетрадка разбита на независимые этапы, чтобы при ошибке перезапускать только проблемную секцию, а не весь pipeline.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def exact_match_rate_from_abs(abs_series, tol=0.0):
    if len(abs_series) == 0:
        return 0.0
    return round((abs_series.fillna(0) <= tol).mean() * 100, 2)

In [ ]:
# Параметры и подключение
report_month = '2026-04-01'
excel_path = '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx'
excel_header = 0

run_invalidate_metadata = False
run_excel_compare = True
show_compare_top = True
save_final_csv = False
output_csv_path = './datamart_month_acquiring_final_2026_04.csv'

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')
snapshot_month_start = month_start

print(f'report_month={report_month}, month_start={month_start}, month_end={month_end}')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

invalidate_tables = [
    'ods_alpha.scd1_agreements', 'ods_alpha.scd1_companies', 'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext', 'cdiul.ext_id_org', 'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals', 'sandbox_ai.shestopalov_terminal_amortization_oneoff',
    'ods_alpha.scd1_trx', 'ods_alpha.scd1_trx_acq', 'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids', 'ods.scd1_z_r2_ip_merchants', 'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix', 'ods.scd1_z_cl_corp', 'ods.scd1_z_depart',
    'ods.scd1_z_branch', 'ods.scd1_z_r2_tariff_plan'
]

if run_invalidate_metadata:
    with imp:
        for t in invalidate_tables:
            imp.execute(f'invalidate metadata {t}')
    print('Invalidate metadata completed')
else:
    print('Invalidate metadata skipped')

## Секция 1. Формирование `final_df` (по шагам)

Ниже каждый этап вынесен в отдельную ячейку:
`01_sa_perimeter` -> `02_cdi_map` -> `03_cft_map` -> `04_operational_metrics` -> `05_transaction_metrics` -> `06_r2_legacy_attrs` -> `07_base_merge` -> `08_agr_fallback` -> `09_actual_tariff_by_agr` -> `10_apply_tariff_fix_and_formulas`.

In [ ]:
# 01_sa_perimeter
sql_sa_perimeter = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_df = imp.fetch(sql_sa_perimeter)

if sa_df is None:
    sa_df = pd.DataFrame()
if not sa_df.empty:
    sa_df['inn'] = sa_df['inn'].map(normalize_inn)
    sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

print(f'SA perimeter rows: {len(sa_df):,}')
display(sa_df.head(3))

In [ ]:
# 02_cdi_map
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

inn_values = sorted([x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_cdi = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm, x_area_resp_norm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  o.x_area_resp_norm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cdi_map_df = imp.fetch(sql_cdi)

if cdi_map_df is None:
    cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
if not cdi_map_df.empty:
    cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
    cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
    allowed = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    cdi_map_df['ssp_ocrm'] = cdi_map_df['ssp_ocrm'].where(cdi_map_df['ssp_ocrm'].isin(allowed), None)
    cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

print(f'CDI rows: {len(cdi_map_df):,}')
display(cdi_map_df.head(3))

In [ ]:
# 03_cft_map
if 'cdi_map_df' not in globals():
    raise RuntimeError('Сначала выполните 02_cdi_map')

cdi_values = sorted([x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cdi_sql_list = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_cft = f"""
select
  cast(e.party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.party_id as string) in ({cdi_sql_list})
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cft_map_df = imp.fetch(sql_cft)

if cft_map_df is None:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
if not cft_map_df.empty:
    cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
    cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
    cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

print(f'CFT rows: {len(cft_map_df):,}')
display(cft_map_df.head(3))

In [ ]:
# 04_operational_metrics
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

if sa_df.empty:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization'])
else:
    n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
    agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

    sql_cmp = f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr, cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({agr_in})
    ),
    m as (
      select sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    ),
    term_active as (
      select m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string) as c_nter
      from m
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = m.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string)
    ),
    retl as (
      select n_agr, count(distinct c_nmrc) as retl_cnt
      from term_active
      group by n_agr
    ),
    term as (
      select n_agr, count(distinct c_nter) as term_cnt
      from term_active
      group by n_agr
    ),
    amort as (
      select ta.n_agr, sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
      from term_active ta
      left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
        on cast(am.c_nter as string) = ta.c_nter
      group by ta.n_agr
    )
    select sa.n_agr, sa.n_cmp_client, r.retl_cnt, t.term_cnt, a.amortization
    from sa_agr sa
    left join retl r on r.n_agr = sa.n_agr
    left join term t on t.n_agr = sa.n_agr
    left join amort a on a.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        cmp_df = imp.fetch(sql_cmp)

if cmp_df is None:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization'])
if not cmp_df.empty:
    cmp_df['n_agr'] = cmp_df['n_agr'].astype(str)
    cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)

print(f'Operational rows: {len(cmp_df):,}')
display(cmp_df.head(3))

In [ ]:
# 05_transaction_metrics
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

if sa_df.empty:
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])
else:
    n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
    agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

    sql_trx = f"""
    with sa_agr as (
      select distinct cast(n_agr as string) as n_agr, cast(n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements
      where cast(n_agr as string) in ({agr_in})
    ),
    trx_base_raw as (
      select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
      from ods_alpha.scd1_trx t
      left join ods_alpha.scd1_base24_fiids fa
        on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
      where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
        and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_base as (
      select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
      from trx_base_raw
      group by n_trx
    ),
    ta_raw as (
      select cast(a.n_trx as string) as n_trx, cast(a.n_agr as string) as n_agr, coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      where cast(a.n_agr as string) in ({agr_in})
    ),
    ta as (
      select n_trx, n_agr, max(n_amt_tax) as n_amt_tax
      from ta_raw
      group by n_trx, n_agr
    ),
    trx_keys as (
      select distinct n_trx
      from ta
    ),
    trx_int_agg as (
      select cast(i.n_trx as string) as n_trx, sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join trx_keys k on k.n_trx = cast(i.n_trx as string)
      group by cast(i.n_trx as string)
    ),
    tj as (
      select ta.n_agr, sa.n_cmp_client, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
      from ta
      join trx_base tb on tb.n_trx = ta.n_trx
      left join sa_agr sa on sa.n_agr = ta.n_agr
    ),
    trx_agg as (
      select
        tj.n_agr,
        count(distinct tj.n_trx) as trx_cnt,
        sum(tj.n_amt_src) as trx_sum,
        sum(tj.n_amt_tax) as commission_from_ops,
        sum(coalesce(i.n_amt_fee, 0.0)) as int_component
      from tj
      left join trx_int_agg i on i.n_trx = tj.n_trx
      group by tj.n_agr
    ),
    active_term_agg as (
      select
        tj.n_cmp_client,
        count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
      from tj
      where tj.n_cmp_client is not null
      group by tj.n_cmp_client
    )
    select
      t.n_agr,
      t.trx_cnt,
      t.trx_sum,
      t.commission_from_ops,
      t.int_component,
      a.n_cmp_client,
      a.active_term_cnt
    from trx_agg t
    left join sa_agr sa on sa.n_agr = t.n_agr
    left join active_term_agg a on a.n_cmp_client = sa.n_cmp_client
    """

    with imp:
        imp.execute('set MEM_LIMIT=16g')
        trx_df = imp.fetch(sql_trx)

if trx_df is None:
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])
if not trx_df.empty:
    trx_df['n_agr'] = trx_df['n_agr'].astype(str)
    trx_df['n_cmp_client'] = trx_df['n_cmp_client'].astype(str)

active_term_df = (
    trx_df[['n_cmp_client', 'active_term_cnt']]
    .dropna(subset=['n_cmp_client'])
    .drop_duplicates(subset=['n_cmp_client'])
    if len(trx_df)
    else pd.DataFrame(columns=['n_cmp_client', 'active_term_cnt'])
)

print(f'Transaction rows: {len(trx_df):,}')
display(trx_df.head(3))

In [ ]:
# 06_r2_legacy_attrs
if 'cft_map_df' not in globals():
    raise RuntimeError('Сначала выполните 03_cft_map')

cft_values = sorted([x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cft_sql_list = ', '.join([f"'{x}'" for x in cft_values]) if cft_values else "''"

if not cft_values:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])
else:
    sql_r2 = f"""
    with r2 as (
      select
        cast(m.id as string) as r2_id,
        cast(m.c_cl_org as string) as cft_id,
        cast(m.c_depart as string) as c_depart,
        cast(m.c_tariff_plan as string) as c_tariff_plan
      from ods.scd1_z_r2_ip_merchants m
      where cast(m.c_cl_org as string) in ({cft_sql_list})
    ),
    fix as (
      select cast(tt.c_tariff_plan as string) as c_tariff_plan, max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_tune tt
      left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
      group by cast(tt.c_tariff_plan as string)
    )
    select
      r2.r2_id,
      r2.cft_id,
      cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
      cast(dep.c_name as string) as vsp_name,
      cast(dep.c_num as string) as vsp_code,
      case
        when br.c_shortlabel is null then null
        when upper(cast(br.c_shortlabel as string)) like '%РФ%'
          then regexp_extract(cast(br.c_shortlabel as string), '^(.*?РФ)', 1)
        else cast(br.c_shortlabel as string)
      end as filial_rf,
      cast(tp.c_name as string) as tariff_name_legacy,
      cast(fix.commission_monthly_fix as decimal(18,2)) as commission_monthly_legacy
    from r2
    left join ods.scd1_z_cl_corp corp on cast(corp.id as string) = r2.cft_id
    left join ods.scd1_z_depart dep on cast(dep.id as string) = r2.c_depart
    left join ods.scd1_z_branch br on cast(br.id as string) = cast(dep.c_filial as string)
    left join ods.scd1_z_r2_tariff_plan tp on cast(tp.id as string) = r2.c_tariff_plan
    left join fix on fix.c_tariff_plan = r2.c_tariff_plan
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        r2_df = imp.fetch(sql_r2)

if r2_df is None:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])

if not r2_df.empty:
    for c in ['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy']:
        r2_df[c] = r2_df[c].astype(str)
    r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')

print(f'R2 legacy rows: {len(r2_df):,}')
display(r2_df.head(3))

In [ ]:
# 07_base_merge
required = ['sa_df', 'cdi_map_df', 'cft_map_df', 'cmp_df', 'trx_df', 'active_term_df', 'r2_df']
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f'Сначала выполните предыдущие шаги: отсутствуют {missing}')

base_df = sa_df.copy()

if not cdi_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cdi_map_df[['inn', 'cdi_id', 'ssp_ocrm']], on='inn', how='left')
else:
    base_df['cdi_id'] = None
    base_df['ssp_ocrm'] = None

if not cft_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
else:
    base_df['cft_id'] = None

if not r2_df.empty and not base_df.empty:
    base_df = base_df.merge(r2_df, on='cft_id', how='left')
else:
    for col in ['r2_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy']:
        base_df[col] = None

if not cmp_df.empty and not base_df.empty:
    base_df = base_df.merge(cmp_df[['n_agr', 'retl_cnt', 'term_cnt', 'amortization']], on='n_agr', how='left')
else:
    for col in ['retl_cnt', 'term_cnt', 'amortization']:
        base_df[col] = None

if not active_term_df.empty and not base_df.empty:
    base_df = base_df.merge(active_term_df[['n_cmp_client', 'active_term_cnt']], on='n_cmp_client', how='left')
else:
    base_df['active_term_cnt'] = None

if not trx_df.empty and not base_df.empty:
    base_df = base_df.merge(trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']], on='n_agr', how='left')
else:
    for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']:
        base_df[col] = None

print(f'base_df rows after merge: {len(base_df):,}')
display(base_df.head(3))

In [ ]:
# 08_agr_fallback
if 'base_df' not in globals():
    raise RuntimeError('Сначала выполните 07_base_merge')

if 'agr_id' not in base_df.columns:
    base_df['agr_id'] = None
if 'r2_id' not in base_df.columns:
    base_df['r2_id'] = None

agr_before_mask = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
base_df['agr_id_source'] = 'sa'
fallback_mask = (~agr_before_mask) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

filled_pct = round(100 * (base_df['agr_id'].notna().mean()), 2) if len(base_df) else 0.0
print(f'agr_id filled after fallback: {filled_pct}%')

In [ ]:
# 09_actual_tariff_by_agr
if 'base_df' not in globals():
    raise RuntimeError('Сначала выполните 08_agr_fallback')

base_df['agr_id_key'] = base_df['agr_id'].map(normalize_agr_q1)
agr_values_fix = sorted([x for x in base_df['agr_id_key'].dropna().astype(str).unique().tolist() if x])
agr_sql_fix = ', '.join([f"'{x}'" for x in agr_values_fix]) if agr_values_fix else "''"

if agr_values_fix:
    sql_actual_tariff_by_agr = f"""
    with agr_actual as (
      select
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_agr as string) as n_agr_actual,
        cast(a.c_agr_number as string) as contract_number_acq,
        cast(a.d_valid_from as date) as d_valid_from_actual,
        cast(a.d_valid_to as date) as d_valid_to_actual,
        row_number() over (
          partition by cast(a.abs_agr_id as string)
          order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
        ) as rn
      from ods_alpha.scd1_agreements a
      where cast(a.abs_agr_id as string) in ({agr_sql_fix})
        and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    merchant_one as (
      select
        cast(m.id as string) as agr_id,
        cast(m.c_tariff_plan as string) as c_tariff_plan,
        row_number() over (
          partition by cast(m.id as string)
          order by cast(m.c_tariff_plan as string) desc
        ) as rn
      from ods.scd1_z_r2_ip_merchants m
      where cast(m.id as string) in ({agr_sql_fix})
    ),
    fix as (
      select cast(tt.c_tariff_plan as string) as c_tariff_plan, max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_tune tt
      left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
      group by cast(tt.c_tariff_plan as string)
    )
    select
      m.agr_id,
      aa.n_agr_actual,
      aa.contract_number_acq,
      aa.d_valid_from_actual,
      aa.d_valid_to_actual,
      m.c_tariff_plan,
      cast(tp.c_name as string) as tariff_name_actual,
      cast(fix.commission_monthly_fix as decimal(18,2)) as commission_monthly_actual
    from merchant_one m
    left join agr_actual aa on aa.agr_id = m.agr_id and aa.rn = 1
    left join ods.scd1_z_r2_tariff_plan tp on cast(tp.id as string) = m.c_tariff_plan
    left join fix on fix.c_tariff_plan = m.c_tariff_plan
    where m.rn = 1
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        actual_tariff_df = imp.fetch(sql_actual_tariff_by_agr)
else:
    actual_tariff_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'c_tariff_plan', 'tariff_name_actual', 'commission_monthly_actual'
    ])

if actual_tariff_df is None:
    actual_tariff_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'c_tariff_plan', 'tariff_name_actual', 'commission_monthly_actual'
    ])

if not actual_tariff_df.empty:
    for c in ['agr_id', 'n_agr_actual', 'contract_number_acq', 'c_tariff_plan', 'tariff_name_actual']:
        actual_tariff_df[c] = actual_tariff_df[c].astype(str).str.strip()

actual_tariff_df = actual_tariff_df.rename(columns={'agr_id': 'agr_id_key'})
print(f'actual_tariff rows: {len(actual_tariff_df):,}')
display(actual_tariff_df.head(3))

In [ ]:
# 10_apply_tariff_fix_and_formulas -> final_df
if 'base_df' not in globals() or 'actual_tariff_df' not in globals():
    raise RuntimeError('Сначала выполните 09_actual_tariff_by_agr')

base_df = base_df.merge(
    actual_tariff_df[['agr_id_key', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual', 'c_tariff_plan', 'tariff_name_actual', 'commission_monthly_actual']],
    on='agr_id_key',
    how='left'
)

base_df['tariff_name'] = base_df['tariff_name_actual'].where(
    base_df['tariff_name_actual'].notna() & (base_df['tariff_name_actual'].astype(str).str.strip() != ''),
    base_df['tariff_name_legacy']
)
base_df['tariff_source'] = base_df['tariff_name_actual'].apply(
    lambda x: 'agr_id_actual' if pd.notna(x) and str(x).strip() else 'legacy_cft'
)
base_df['commission_monthly'] = pd.to_numeric(base_df['commission_monthly_actual'], errors='coerce').where(
    pd.to_numeric(base_df['commission_monthly_actual'], errors='coerce').notna(),
    pd.to_numeric(base_df['commission_monthly_legacy'], errors='coerce')
)

commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)
retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)

base_df['commission_total'] = commission_from_ops_num + commission_monthly_num
base_df['aur'] = retl_cnt_num * 1926
base_df['chod'] = base_df['commission_total'] + int_component_num
base_df['fin_result'] = base_df['chod'] - pd.to_numeric(base_df['aur'], errors='coerce').fillna(0) - amortization_num
base_df['report_month'] = report_month_label
base_df['snapshot_month_start'] = snapshot_month_start

mvp_columns = [
    'report_month', 'snapshot_month_start', 'inn', 'company_name',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
    'cdi_id', 'ssp_ocrm', 'cft_id', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code',
    'tariff_name', 'tariff_source',
    'retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result'
]

for col in mvp_columns:
    if col not in base_df.columns:
        base_df[col] = None

for col in ['trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total', 'aur', 'amortization', 'chod', 'fin_result']:
    base_df[col] = base_df[col].map(to_decimal_or_none)

for col in ['retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt']:
    base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

final_df = base_df[mvp_columns].copy()

if final_df is None or len(final_df) == 0:
    raise RuntimeError('QC fail: final_df пустой')
if 'tariff_name' not in final_df.columns:
    raise RuntimeError('QC fail: отсутствует tariff_name')

tariff_actual_rows = int((final_df['tariff_source'] == 'agr_id_actual').sum()) if len(final_df) else 0
tariff_actual_pct = round(100 * tariff_actual_rows / len(final_df), 2) if len(final_df) else 0.0

print(f'final_df rows: {len(final_df):,}')
print(f'tariff_source=agr_id_actual: {tariff_actual_rows:,} ({tariff_actual_pct}%)')
display(final_df.head(5))

## Секция 2. Сравнение `final_df` и Excel (`cmp` + KPI)

Секция независима от шагов выше, кроме наличия `final_df`.